# Query Planning: Parallel Retrieval and Dependent Comparison

| Field | Value |
|---|---|
| Stage | Autonomous RAG patterns |
| Difficulty | Advanced |
| Status | Complete |
| Requires network/API | No |
| Last reviewed | 2026-09-25 |

Callout - Key idea:
A query plan is a dependency graph: independent retrievals may run in parallel, but comparison waits for both typed results.

## 30-Second Summary

This notebook plans a retention comparison between Atlas and Beacon. Two retrieval tasks are independent; a comparison task depends on both. Coverage and dependency checks prevent premature synthesis.

## Why This Matters

Basic decomposition lists questions. Planning also states execution order, output schema, and dependencies needed for derived conclusions.

## Scope

| Covers | Does not cover |
|---|---|
| DAG dependencies, parallel-ready tasks, typed results, comparison | Async runtime benchmark, dynamic LLM planner, large workflow engine |


## Mental Model

```text
retrieve Atlas --\
                  compare longer -> synthesize
retrieve Beacon -/
```


In [1]:
SOURCES = {
    "atlas": {"service": "Atlas", "retention_days": 30, "source": "atlas-policy"},
    "beacon": {"service": "Beacon", "retention_days": 90, "source": "beacon-policy"},
}
question = "Compare Atlas and Beacon log retention and identify which is longer."


## How It Works

Each node declares dependencies and an output key. The two retrieval tasks have none and can run concurrently; comparison depends on both outputs and emits a derived result with source provenance.


## Baseline

A single top-1 lookup returns Atlas only, so no valid comparison is possible.


In [2]:
baseline = {"atlas": SOURCES["atlas"]}
baseline_can_compare = {"atlas", "beacon"} <= set(baseline)
baseline_can_compare


False

## Technique Implementation

The plan is a small DAG represented as data. A topological check ensures dependencies are satisfied before execution.


In [3]:
plan = [
    {"id": "get_atlas", "depends_on": [], "output": "atlas"},
    {"id": "get_beacon", "depends_on": [], "output": "beacon"},
    {"id": "compare", "depends_on": ["atlas", "beacon"], "output": "comparison"},
]
results = {}
for task in plan:
    assert set(task["depends_on"]) <= set(results), f"unsatisfied dependencies for {task['id']}"
    if task["id"] == "get_atlas": results["atlas"] = SOURCES["atlas"]
    elif task["id"] == "get_beacon": results["beacon"] = SOURCES["beacon"]
    else:
        longer = max((results["atlas"], results["beacon"]), key=lambda item: item["retention_days"])
        results["comparison"] = {"longer_service": longer["service"], "days": longer["retention_days"]}
results


{'atlas': {'service': 'Atlas', 'retention_days': 30, 'source': 'atlas-policy'},
 'beacon': {'service': 'Beacon',
  'retention_days': 90,
  'source': 'beacon-policy'},
 'comparison': {'longer_service': 'Beacon', 'days': 90}}

## Controlled Experiment

We compare source coverage and verify that the derived comparison uses both typed values and retains both citations.


In [4]:
citations = sorted((results["atlas"]["source"], results["beacon"]["source"]))
answer = (
    f"Atlas retains logs for {results['atlas']['retention_days']} days [atlas-policy]; "
    f"Beacon retains them for {results['beacon']['retention_days']} days [beacon-policy]. "
    f"{results['comparison']['longer_service']} is longer."
)
experiment = {"baseline_can_compare": baseline_can_compare, "planned_can_compare": "comparison" in results, "citations": citations, "answer": answer}
experiment


{'baseline_can_compare': False,
 'planned_can_compare': True,
 'citations': ['atlas-policy', 'beacon-policy'],
 'answer': 'Atlas retains logs for 30 days [atlas-policy]; Beacon retains them for 90 days [beacon-policy]. Beacon is longer.'}

## Evaluation

The baseline cannot compare. The DAG retrieves both sources, computes `Beacon` as longer at **90 days**, and cites both policies. The plan demonstrates dependencies; it does not benchmark actual parallel speedup.


In [5]:
assert not experiment["baseline_can_compare"] and experiment["planned_can_compare"]
assert results["comparison"] == {"longer_service": "Beacon", "days": 90}
assert experiment["citations"] == ["atlas-policy", "beacon-policy"]
assert all(f"[{source}]" in answer for source in experiment["citations"])
print("Query-plan dependency checks passed.")


Query-plan dependency checks passed.


## Decision Guide

| Relationship | Execution |
|---|---|
| Independent facts | Parallel |
| Entity discovered first | Sequential |
| Comparison | Retrieve same schema for every entity |
| Missing dependency | Stop/partial result with caveat |


## Failure Modes and Debugging

| Symptom | Cause | Fix |
|---|---|---|
| Comparison uses one side | Dependency omitted | Typed dependency check |
| Apples-to-oranges values | Schemas/units differ | Normalize before compare |
| Slow despite independence | Serial executor | Parallel task group |
| Partial result called complete | No coverage gate | Required outputs checklist |


## Production Notes

### Observability
Trace DAG, task status, dependency waits, source IDs, units, latency, and partial failures.

### Safety and Guardrails
All parallel branches inherit authorization; do not merge cross-tenant evidence.

### Latency and Cost
Bound fan-out and parallelize only independent tasks.


## Practice

Add a third service with retention reported in months and insert a normalization dependency before comparison.

## Recall

Toggle - Recall: What distinguishes planning from a subquery list?
Dependencies, output schemas, and execution order.

Toggle - Recall: When is parallelism safe?
When tasks do not depend on each other's outputs.

## Sources

- [LangGraph orchestrator-worker workflow](https://docs.langchain.com/oss/python/langgraph/workflows-agents)
- Repository-owned synthetic policy data

## Review Log

| Date | Status | Confidence | Next review focus |
|---|---|---|---|
| 2026-09-25 | Complete; executed and visually reviewed | High for the explicit DAG fixture | Add async execution and partial-failure policy |
